## Imports

In [2]:
from datasets import load_dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
import torch
import pandas as pd

In [17]:
base_path = "/content/drive/MyDrive/pfe/gpt2"
data_path = 'https://raw.githubusercontent.com/abdelhaqelamraoui/pfe_product_description_generation/refs/heads/main/data/data_prepared.csv'

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### A. Load Dataset

In [5]:
df = pd.read_csv(data_path)

In [6]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)
train_test_split = dataset.train_test_split(test_size=0.2)
train_data = train_test_split['train']
test_data = train_test_split['test']

### B. Tokenize Data

In [7]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 doesn't have a pad token, so we set it to eos_token

def tokenize_function(examples):
    # Combine input and output for GPT-2 (since it's a causal LM, it predicts next token)
    texts = [f"Input: {inp} Output: {out}" for inp, out in zip(examples['input'], examples['output'])]
    encodings = tokenizer(
        texts,
        max_length=256,  # Increased max_length to accommodate input + output
        truncation=True,
        padding='max_length',
        return_tensors='pt'
    )
    return {
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'labels': encodings['input_ids']  # For causal LM, labels are the same as input_ids
    }

tokenized_train = train_data.map(tokenize_function, batched=True)
tokenized_test = test_data.map(tokenize_function, batched=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Map:   0%|          | 0/6204 [00:00<?, ? examples/s]

Map:   0%|          | 0/1551 [00:00<?, ? examples/s]

### C. Fine-Tune GPT-2

In [8]:
model = GPT2LMHeadModel.from_pretrained('gpt2')

training_args = TrainingArguments(
    output_dir=f'{base_path}/gpt2-product-description',
    per_device_train_batch_size=4,
    num_train_epochs=3,
    save_steps=10_000,
    save_total_limit=2,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy='steps',
    eval_steps=500,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

trainer.train()

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

...

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shepherd-boy1212 (shepherd-boy1212-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
500,1.436500,1.335124
1000,1.355300,1.270580
1500,1.328900,1.230847
2000,1.208900,1.209745
2500,1.161000,1.195068
3000,1.182600,1.181467
3500,1.122800,1.176228
4000,1.143900,1.171350
4500,1.144600,1.168529


TrainOutput(global_step=4653, training_loss=1.2478149605904143, metrics={'train_runtime': 1978.082, 'train_samples_per_second': 9.409, 'train_steps_per_second': 2.352, 'total_flos': 2431583649792000.0, 'train_loss': 1.2478149605904143, 'epoch': 3.0})

### D. Generate Descriptions

In [18]:
import torch

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

def generate_description(input_text, max_length=256, num_beams=4):
    # Format input text
    input_text = f'Input: {input_text} Output:'
    inputs = tokenizer(input_text, return_tensors='pt', max_length=128, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Generate description
    outputs = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_length=max_length,
        num_beams=num_beams,
        no_repeat_ngram_size=2,
        early_stopping=True
    )

    # Decode and clean output
    description = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the output part after 'Output:'
    description = description.split('Output:')[-1].strip().capitalize()
    return description

# Example
input_text = 'Style: Industrial | Material: Metal | Color: Gray | Dimensions: 30L x 20W x 15H'
print(generate_description(input_text))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Add a touch of elegance to your living space with this stylish and functional coffee table crafted from high quality mdf this table is built to last ensuring it will last for years to come


### E. Save Model

In [10]:
# Save model
model.save_pretrained(f'{base_path}/fine-tuned-gpt2')
tokenizer.save_pretrained(f'{base_path}/fine-tuned-gpt2')

# Load later
model = GPT2LMHeadModel.from_pretrained(f'{base_path}/fine-tuned-gpt2')
tokenizer = GPT2Tokenizer.from_pretrained(f'{base_path}/fine-tuned-gpt2')

In [19]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


model = GPT2LMHeadModel.from_pretrained(f'{base_path}/fine-tuned-gpt2', local_files_only=True).to(device)
tokenizer = GPT2Tokenizer.from_pretrained(f'{base_path}/fine-tuned-gpt2', local_files_only=True)
tokenizer.pad_token = tokenizer.eos_token

def generate_product_description(input_features, max_length=256, num_beams=4):
    """
    Generate product description from input features
    """
    input_text = f'Input: {input_features} Output:'

    # Tokenize inputs
    inputs = tokenizer(
        input_text,
        max_length=128,
        truncation=True,
        return_tensors='pt'
    )

    # Move inputs to the same device as the model
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Generate description
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=max_length,
            num_beams=num_beams,
            no_repeat_ngram_size=2,
            early_stopping=True
        )

    # Decode and clean output
    description = tokenizer.decode(outputs[0], skip_special_tokens=True)
    description = description.split('Output:')[-1].strip().capitalize()

    return description

# Example usage
input_features = 'Style: Industrial | Material: Metal | Color: Gray | Dimensions: 30L x 20W x 15H | Features: Rust-proof, Wall-mounted'
generated_description = generate_product_description(input_features)

print('Input Features:')
print(input_features)
print('\nGenerated Description:')
print(generated_description)


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Input Features:
Style: Industrial | Material: Metal | Color: Gray | Dimensions: 30L x 20W x 15H | Features: Rust-proof, Wall-mounted

Generated Description:
Add a touch of elegance to your living space with this stylish and functional coffee table made of high quality mdf board this table is built to last ensuring it will last for years to come the sturdy metal frame ensures stability and durability while the adjustable height allows you to customize the height to fit your needs
